# ASAP7 regression residualization — aligned session-time version

This notebook is intentionally short. The helper module `asap7_regression_tools.py` handles the verbose alignment, loading, and regression functions.

**Time convention used throughout**

All streams are mapped onto **session-relative HARP seconds**:

```text
session_time_sec = absolute_harp_time - first_encoder_harp_time
```

For the session files you attached, this gives:

- encoder/running: starts at `0 s`
- imaging epoch: starts at `~6.386656 s`
- image/change/omission onsets from voltage QC: already session-relative
- voltage H5 timebase: auto-detected as either session-relative or imaging-relative and corrected accordingly


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

# Put asap7_regression_tools.py next to this notebook.
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import vip_slap2_analysis.voltage.asap7_regression_tools as ar

plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", 80)

import warnings
from vip_slap2_analysis.io.session_registry import VIPSessionRegistry
from vip_slap2_analysis.utils.utils import save_figure
from IPython.display import display, HTML

display(HTML("<style>.container { width:100% !important; }</style>"))
warnings.filterwarnings("default")

In [ ]:
def plot_r2_summary_fixed(r2_df):
    order = [m for m in ["drift", "drift_running", "drift_stimulus", "full"]
             if m in set(r2_df["model"])]
    dmds = sorted(r2_df["dmd"].unique())

    fig, ax = plt.subplots(figsize=(9, 4))

    x = np.arange(len(order))
    width = 0.8 / max(1, len(dmds))

    for j, dmd_name in enumerate(dmds):
        vals = [
            r2_df.loc[
                (r2_df["dmd"] == dmd_name) & (r2_df["model"] == model_name),
                "r2"
            ].to_numpy(dtype=float)
            for model_name in order
        ]

        means = [np.nanmean(v) if len(v) else np.nan for v in vals]
        sems = [
            np.nanstd(v, ddof=1) / np.sqrt(max(np.isfinite(v).sum(), 1))
            if np.isfinite(v).sum() > 1 else 0
            for v in vals
        ]

        xpos = x - 0.4 + width / 2 + j * width

        ax.bar(
            xpos,
            means,
            width=width,
            yerr=sems,
            capsize=3,
            alpha=0.8,
            label=dmd_name,
        )

        for i, v in enumerate(vals):
            if len(v) == 0:
                continue
            jitter = np.linspace(-0.25 * width, 0.25 * width, len(v))
            ax.plot(
                np.full(len(v), xpos[i]) + jitter,
                v,
                "o",
                ms=3,
                alpha=0.55,
            )

    ax.set_xticks(x)
    ax.set_xticklabels(order, rotation=30, ha="right")
    ax.set_ylabel("Held-out R²")
    ax.set_title("Regression model performance by DMD")
    ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)

    plt.tight_layout()
    return fig, ax


# Monkey-patch the helper module for this notebook session.
ar.plot_r2_summary = plot_r2_summary_fixed

In [ ]:
%matplotlib inline

## Load the data asset objects

In [ ]:
today_str = datetime.today().strftime('%Y-%m-%d')

BASE_PATH = Path(r'\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics')
SAVE_PATH = Path(r'C:\Users\andrew.shelton\Dropbox\allen institute\Documents\Presentations\OPhys\Lab_Meetings\2026-07-28_OPhys_LabMeetingV\figures\voltage_plots')

TARGET_MICE = [
    826031,
    826032
]

PARADIGMS = ["change_detection_passive"]
EXCLUDE_SESSION_TYPES = ["expression_check", "volume_imaging"]

# Voltage extraction writes files like:
#   voltage_session_traces_dff_robust_f0_trial.h5
# This variant controls the filename suffix. The plotted dataset is SIGNAL below.
TRACE_VARIANT = "dff_robust_f0_trial"
SIGNAL = "dff"      # one of: "raw_f", "f0", "dff"

# Optional direct override. Leave as None to resolve from asset.derived_dir / "voltage".
SESSION_TRACE_H5 = None

In [ ]:
registry = VIPSessionRegistry.from_basepath(BASE_PATH)

process_df = registry.sessions(
    subject_ids=TARGET_MICE,
    exclude_session_types=EXCLUDE_SESSION_TYPES,
    paradigms=PARADIGMS,
)

assets = [registry.resolve_assets(row) for _, row in process_df.iterrows()]

print(f"Found {len(assets)} candidate sessions.")
display(process_df)

## 1. Configuration

Use `assets[SESSION_INDEX]` when available. Manual paths override asset-derived paths and are useful for debugging.


In [ ]:
SESSION_INDEX = 0

# Leave these as None when using your normal `assets` object.
MANUAL_TRACE_H5 = None
MANUAL_QC_DIR = None
MANUAL_ENCODER_PKL = None
MANUAL_HARP_DF_CSV = None
MANUAL_BONSAI_EVENT_LOG_CSV = None

# Voltage/session trace configuration.
TRACE_VARIANT = "dff_robust_f0_trial"
SIGNAL_NAME = "dff"
REG_TARGET_FS_HZ = 100.0
VALID_ROIS_ONLY = True
MAX_SECONDS = None  # e.g. 300 for a smoke test

# Running-wheel constants.
TICKS_PER_REVOLUTION = 8192
WHEEL_RADIUS_CM = 8.225

# Timebase behavior.
# "auto" should be correct for both session-relative and imaging-relative voltage H5 timebases.
VOLTAGE_TIME_MODE = "auto"

# Event extraction.
EVENT_SOURCE_DMD = "auto"  # uses DMD1 if present


## 2. Resolve paths and align behavior/stimulus/imaging

This is the critical alignment section. It should show running from 0 to ~session duration, imaging starting slightly later, and event times overlapping the imaging epoch.


In [ ]:
asset = assets[SESSION_INDEX] if "assets" in globals() else None

paths = ar.resolve_session_paths(
    asset,
    trace_h5=MANUAL_TRACE_H5,
    qc_dir=MANUAL_QC_DIR,
    encoder_pkl=MANUAL_ENCODER_PKL,
    harp_df_csv=MANUAL_HARP_DF_CSV,
    bonsai_event_log_csv=MANUAL_BONSAI_EVENT_LOG_CSV,
    trace_variant=TRACE_VARIANT,
)

display(pd.DataFrame({
    "item": ["trace_h5", "qc_dir", "encoder_pkl", "harp_df_csv", "bonsai_event_log_csv"],
    "path": [paths.trace_h5, paths.qc_dir, paths.encoder_pkl, paths.harp_df_csv, paths.bonsai_event_log_csv],
}))


In [ ]:
ctx = ar.build_timing_context(
    paths,
    ticks_per_revolution=TICKS_PER_REVOLUTION,
    wheel_radius_cm=WHEEL_RADIUS_CM,
    event_source_dmd=EVENT_SOURCE_DMD,
)

summary = ar.alignment_summary_table(ctx)
display(summary)

print("session_harp_t0_abs:", ctx["session_harp_t0_abs"])
print("HARP_df summary:", ctx["harp_summary"])
print("event metadata:", ctx["event_meta"])

ar.assert_timing_alignment(ctx)
print("Alignment checks passed.")


In [ ]:
# First 90 seconds after imaging onset.
fig, axes = ar.plot_alignment_overview(ctx, duration_sec=1800)
plt.show()


## 3. Load voltage on the aligned session timebase

The helper auto-detects whether `timebase_sec` is already session-relative or needs the imaging epoch start added. Inspect `time_mode_used` below. If this is wrong, override with:

```python
VOLTAGE_TIME_MODE = "already_session"
# or
VOLTAGE_TIME_MODE = "imaging_relative"
```


In [ ]:
voltage = ar.load_voltage_lowfreq(
    paths.trace_h5,
    ctx,
    signal_name=SIGNAL_NAME,
    target_fs_hz=REG_TARGET_FS_HZ,
    valid_rois_only=VALID_ROIS_ONLY,
    max_seconds=MAX_SECONDS,
    voltage_time_mode=VOLTAGE_TIME_MODE,
)

rows = []
for dmd, d in voltage["dmd"].items():
    rows.append({
        "dmd": dmd,
        "n_rois": d["X"].shape[0],
        "n_samples": d["X"].shape[1],
        "fs_hz": d["fs_hz"],
        "time_start_sec": float(d["time_sec"][0]),
        "time_end_sec": float(d["time_sec"][-1]),
        "time_mode_used": d["time_mode_used"],
    })
display(pd.DataFrame(rows))


In [ ]:
# Verify that running interpolates onto voltage time; this should not be empty.
for dmd, d in voltage["dmd"].items():
    rt = ar.interpolate_running_to_time(ctx, d["time_sec"])
    print(dmd, "finite speed fraction:", np.mean(np.isfinite(rt["speed_cm_s"])))
    print(dmd, "median speed:", np.nanmedian(rt["speed_cm_s"]), "cm/s")


In [ ]:
# Visual alignment of voltage population, running, and events over one short window.
dmd0 = sorted(voltage["dmd"].keys())[0]
t = voltage["dmd"][dmd0]["time_sec"]
Xz = ar.robust_zscore_rows(voltage["dmd"][dmd0]["X"])
pop = np.nanmean(Xz, axis=0)
run_t = ar.interpolate_running_to_time(ctx, t)

t0 = max(float(ctx["epochs"]["start_time_sec"].min()), float(t[0])) + 5
t1 = t0 + 120
m = (t >= t0) & (t <= t1)

fig, ax = plt.subplots(figsize=(12, 4))

# Voltage trace
v_lines = ax.plot(
    t[m],
    pop[m],
    lw=1,
    label=f"{dmd0} population voltage",
)

# Running trace on second y-axis
ax2 = ax.twinx()
s_lines = ax2.plot(
    t[m],
    run_t.loc[m, "speed_cm_s"],
    lw=1,
    alpha=0.6,
    label="running speed",color='tab:orange'
)

# Event markers
change_line = None
omission_line = None

for ev in ctx["events"]["change"]:
    if t0 <= ev <= t1:
        change_line = ax.axvline(
            ev,
            ls="--",
            lw=0.8,
            alpha=0.8,
            label="change" if change_line is None else None,color='blue'
        )

for ev in ctx["events"]["omission"]:
    if t0 <= ev <= t1:
        omission_line = ax.axvline(
            ev,
            ls=":",
            lw=1.2,
            alpha=0.9,
            label="omission" if omission_line is None else None,color='red'
        )

ax.set_xlabel("session-relative HARP time (s)")
ax.set_ylabel("population voltage, robust z")
ax2.set_ylabel("running speed (cm/s)")
ax.set_title("Final alignment sanity check")

ax.spines[["top"]].set_visible(False)
ax2.spines[["top"]].set_visible(False)

# Combined legend from both y-axes plus event markers
handles1, labels1 = ax.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()

ax.legend(
    handles1 + handles2,
    labels1 + labels2,
    loc="best",
    frameon=False,
)

plt.show()

## 4. Build regression design

Predictors:

- slow drift basis
- lagged running velocity/speed/acceleration
- image flash kernels
- change kernels
- omission kernels

This first pass uses dense design matrices and in-sample ridge fits for residualization. Treat R² as descriptive, not cross-validated model performance.


In [ ]:
DESIGN_PARAMS = dict(
    n_drift_basis=12,
    running_lags_sec=(-1.0, -0.5, 0.0, 0.5, 1.0),
    event_window=(-0.5, 1.5),
    event_bin_width=0.05,
)

# Use the first DMD timebase for the design. In normal session-long H5s DMDs share time.
t_model = voltage["dmd"][sorted(voltage["dmd"].keys())[0]]["time_sec"]
X_design, design_meta = ar.build_regression_design(t_model, ctx, **DESIGN_PARAMS)

print("Design shape:", X_design.shape)
display(design_meta.groupby("group").size().rename("n_predictors").reset_index())
display(design_meta.head())


## 5. Fit ROI-wise regression models

Models:

- `drift`
- `drift_running`
- `drift_stimulus`
- `full`


In [ ]:
RIDGE_ALPHA = 10.0

fits = {}
for dmd, d in voltage["dmd"].items():
    if not np.allclose(d["time_sec"], t_model, atol=0.5 / d["fs_hz"]):
        raise ValueError(f"{dmd} timebase does not match design timebase.")
    fits[dmd] = ar.fit_ridge_models(
        X_design,
        d["X"],
        design_meta,
        alpha=RIDGE_ALPHA,
    )

r2_df = ar.summarize_r2(fits)
display(r2_df.groupby(["dmd", "model"])["r2"].agg(["count", "mean", "median", "std"]).reset_index())
fig, ax = ar.plot_r2_summary(r2_df)
ax.set_ylabel("In-sample R²")
ax.set_title("Regression model fit by DMD")
plt.show()


## 6. Inspect residualization in the frequency domain

The main question is whether image-cadence/task-band power and DMD population coherence drop after removing running/stimulus/drift predictors.


In [ ]:
MODEL_FOR_RESIDUALS = "full"

fig, ax = ar.plot_raw_vs_residual_psd(
    voltage,
    fits,
    model=MODEL_FOR_RESIDUALS,
    fmax=1000.0,
    window_sec=256.0,
)
plt.show()

fig, ax = ar.plot_population_coherence_before_after(
    voltage,
    fits,
    model=MODEL_FOR_RESIDUALS,
    fmax=1000.0,
    window_sec=256.0,
)
plt.show()


## 7. Save residual outputs

The saved `.npz` includes raw downsampled traces, predictions, residuals, ROI IDs, and aligned session-relative time vectors.


In [ ]:
if asset is not None and getattr(asset, "derived_dir", None) is not None:
    OUT_DIR = Path(asset.derived_dir) / "regression_residuals"
else:
    OUT_DIR = Path("regression_residuals")

out_dir = ar.save_regression_outputs(
    OUT_DIR,
    voltage,
    fits,
    design_meta,
    r2_df,
    model=MODEL_FOR_RESIDUALS,
)
print("Saved:", out_dir)


### Preliminary interpretation
On the full continuous voltage traces, slow session-scale fluctuations explain more variance than the current running or stimulus-timing regressors. Running adds a small but real-looking amount. Generic image/change/omission timing adds surprisingly little. The dominant DMD-shared voltage structure therefore is not well explained by this simple linear event-timing model.

In [ ]:
# Quantify incremental R² by ROI and DMD.
wide_r2 = (
    r2_df
    .pivot_table(index=["dmd", "roi_index"], columns="model", values="r2")
    .reset_index()
)

wide_r2["running_delta_vs_drift"] = wide_r2["drift_running"] - wide_r2["drift"]
wide_r2["stimulus_delta_vs_drift"] = wide_r2["drift_stimulus"] - wide_r2["drift"]
wide_r2["full_delta_vs_drift"] = wide_r2["full"] - wide_r2["drift"]
wide_r2["full_delta_vs_drift_running"] = wide_r2["full"] - wide_r2["drift_running"]
wide_r2["full_delta_vs_drift_stimulus"] = wide_r2["full"] - wide_r2["drift_stimulus"]

display(
    wide_r2
    .groupby("dmd")
    [["drift", "running_delta_vs_drift", "stimulus_delta_vs_drift", 
      "full_delta_vs_drift", "full_delta_vs_drift_running"]]
    .agg(["mean", "median", "std"])
)